# Notebook para probar los modelos en el dataset de validación (comparación con modelos Benchmark)

*Aquí se detalla el procedimiento seguido para el experimento 4*

In [107]:
import pandas as pd
import numpy as np

Cargar los datos

In [108]:
data=pd.read_csv("creacionModelosBenchMark/validation.csv")

# Glove

In [109]:
import gensim
import numpy as np
import pandas as pd
from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.corpus import stopwords
import nltk
nltk.download('punkt')
nltk.download('stopwords')

from nltk.tokenize import sent_tokenize

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\angel\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\angel\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [110]:
def load_glove_model(glove_file):
    print("Loading GloVe Model...")
    glove_model = {}
    with open(glove_file, 'r', encoding="utf-8") as f:
        for line in f:
            split_line = line.split()
            word = split_line[0]
            embedding = np.array([float(val) for val in split_line[1:]])
            glove_model[word] = embedding
    print("Done.", len(glove_model), "words loaded!")
    return glove_model

glove_file = 'glove_6B/glove-sbwc.i25.vec'
glove_model = load_glove_model(glove_file)

Loading GloVe Model...
Done. 855380 words loaded!


In [111]:
def tokenizar(data):
    respuestas = list(data['respuestas'].values)
    #counter=0 Este counter se utiliza si se desea limitar el número de oraciones en cada respuesta
    indice_extras=[]
    respuestas_token = []
    for i in respuestas:
        #Tokenize the text into sentences
        sentences = sent_tokenize(i)
        #if len(sentences)<3 or len(sentences)>100:
        #   indice_extras.append(counter)
        #else:
            # Tokenize each sentence into words
        tokenized_sentences = [word_tokenize(sentence) for sentence in sentences]
        respuestas_token.append(tokenized_sentences)
    
    stop_words = set(stopwords.words('spanish'))

    def sentence_embedding(sentence_tokens, embeddings, stop_words):
        embedding_dim = 300  # GloVe 50d
        sentence_vector = np.zeros(embedding_dim)
        word_count = 0
        
        for word in sentence_tokens:
            word = word.lower()
            if word not in stop_words and word in embeddings:
                sentence_vector += embeddings[word]
                word_count += 1
                
        if word_count > 0:
            sentence_vector /= word_count  # Optionally, normalize by number of words
        
        return sentence_vector
        # Calculate embeddings for each sentence, ignoring stopwords
    gloVe_Embedding=[]
    for tokenized_sentences in respuestas_token:

        sentence_embeddings = [sentence_embedding(sentence, glove_model, stop_words) for sentence in tokenized_sentences]
        sentence_embeddings = np.array(sentence_embeddings)
        gloVe_Embedding.append(sentence_embeddings)
    
    return gloVe_Embedding

In [112]:
embedding=tokenizar(data)

# TDA

In [113]:
import ripser
from persim import plot_diagrams, PersistenceImager
import matplotlib.pyplot as plt
import gensim
import numpy as np
import pandas as pd

In [114]:
def generarImagenes(gloVe_Embedding): 
    lista_vectores = []
    for i in range(len(gloVe_Embedding)):
        ripserperiod = ripser.ripser(gloVe_Embedding[i])["dgms"]
        h0_diagram = ripserperiod[0].copy()
        h0_diagram = h0_diagram[np.isfinite(h0_diagram).all(axis=1)]
        lista_vectores.append(h0_diagram)

    #ignore warnings
    import warnings
    warnings.filterwarnings("ignore")

        # Generate the images for all the persistence diagrams at once to assure the same pixel size
    # Manually set birth and persistence ranges based on your data
    birth_range = (0, 0.5)  # Adjust these values as per your data
    pers_range = (0, 3)   # Adjust these values as per your data

    # Initialize PersistenceImager
    pimgr = PersistenceImager(pixel_size=0.01, birth_range=birth_range, pers_range=pers_range)
    pimgr.kernel_params = {'sigma': 0.01}
    pdgms = lista_vectores
    #pimgr.fit(lista_vectores, skew=True)
    pimgs = pimgr.transform(pdgms,skew=True)

    return pimgs

In [115]:
imagenes=generarImagenes(embedding)


In [116]:
import pandas as pd
import numpy as np
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import pickle

In [117]:
def probar_modeloLasso(imagenes, labels, tamano=0, return_predicciones=False):
    #Convert imagenes array from 3D to 2D
    lol=[]
    for i in range(len(imagenes)):
        lol.append(imagenes[i].flatten())
    imagenes=np.array(lol)
    
    #Cargar el modelo desde pickle
    with open(f'creacionModelosBenchMark/modeloLASSO_{tamano}.pkl', 'rb') as file:
        modelo = pickle.load(file)
    
    # Predecir las etiquetas para el conjunto de imágenes
    predicciones = modelo.predict(imagenes)
    # Calcular métricas de evaluación
    accuracy = accuracy_score(labels, predicciones)
    if return_predicciones:
        return predicciones
    else:
        print(f"Accuracy: {accuracy:.2f}")
        return accuracy

    


In [118]:
probar_modeloLasso(imagenes, data['ai'], tamano='chicos')
probar_modeloLasso(imagenes, data['ai'], tamano='grandes')

Accuracy: 0.49
Accuracy: 0.78


0.782608695652174

# LSTM

In [119]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Bidirectional, LSTM, Dense, Masking
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import BinaryCrossentropy
# Load the keras model
from keras.models import load_model


In [120]:
def probar_modeloLSTM(val_glove, labels, tamano=0, return_predicciones=False):
    max_timesteps = 8  # Define the maximum length for padding

    X = val_glove

    # Pad the sequences to have the same length
    X_padded = pad_sequences(X, maxlen=max_timesteps, dtype='float32', padding='post', truncating='post')

   # Cargar el modelo desde keras, como archivo local
    modelo = load_model(f'creacionModelosBenchMark/modeloLSTM_{tamano}.keras')


    # Predecir las etiquetas para el conjunto de imágenes
    predicciones = modelo.predict(X_padded)
    predicciones = np.round(predicciones).astype(int).flatten()

    # Calcular métricas de evaluación
    accuracy = accuracy_score(labels, predicciones)
    
    if return_predicciones:
        return predicciones
    else:
        print(f"Accuracy: {accuracy:.2f}")
        return accuracy
    


In [121]:
probar_modeloLSTM(embedding, data['ai'], tamano='chicos')
probar_modeloLSTM(embedding, data['ai'], tamano='grandes')

5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 125ms/step
Accuracy: 0.82
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 131ms/step
Accuracy: 0.98


0.9782608695652174

# TDA + LSTM

In [122]:
def probar_modeloLasso_LSTM(imagenes, val_glove,labels, tamano=0):
    
    prediccionesLasso=probar_modeloLasso(imagenes, labels, tamano=tamano, return_predicciones=True)
    prediccionesLSTM=probar_modeloLSTM(val_glove, labels, tamano=tamano, return_predicciones=True)

    w=4
    pred_final=[]
    for i in range(len(prediccionesLasso)):
        pred_final.append((prediccionesLasso[i]+prediccionesLSTM[i]*w)/(1+w))
    # Redondear las predicciones finales a 0 o 1
    pred_final = np.round(pred_final).astype(int)
    print(f"Accuracy: {accuracy_score(labels, pred_final):.2f}")
    # Combine the predictions from both model
    return accuracy_score(labels, pred_final)


In [123]:
probar_modeloLasso_LSTM(imagenes, embedding, data['ai'], tamano='chicos')
probar_modeloLasso_LSTM(imagenes, embedding, data['ai'], tamano='grandes')

5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 101ms/step
Accuracy: 0.82
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 98ms/step
Accuracy: 0.98


0.9782608695652174

# BERT

In [124]:
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments
from datasets import load_dataset
import torch
import tensorflow as tf

from tensorflow.keras import layers, models

import pandas as pd

In [125]:
model = BertForSequenceClassification.from_pretrained("creacionModelosBenchMark/modeloBERT_chicos")

In [ ]:
def probar_modeloBert(df, tamano=0):

   # Load model and tokenizer from the saved folder
    model = BertForSequenceClassification.from_pretrained(f"creacionModelosBenchMark/modeloBERT_{tamano}")
    tokenizer = BertTokenizer.from_pretrained(f"creacionModelosBenchMark/modeloBERT_{tamano}")
    

    model.eval()  # Set model to evaluation mode

    texts = df['respuestas'].tolist()  

    encodings = tokenizer(texts, truncation=True, padding=True, return_tensors='pt')

    
    with torch.no_grad():  # Disable gradient calculation for inference
        outputs = model(**encodings)
        logits = outputs.logits
        predictions = torch.argmax(logits, dim=1).numpy()


    true_labels = df['ai'].values
    accuracy = accuracy_score(true_labels, predictions)
    print(f"Validation Accuracy: {accuracy:.4f}")
    return accuracy

In [127]:
probar_modeloBert(data, tamano='chicos')
probar_modeloBert(data, tamano='grandes')

Validation Accuracy: 0.9275
Validation Accuracy: 0.9783


0.9782608695652174

# Transformers


In [128]:

from tensorflow.keras import layers, models
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.sequence import pad_sequences
import tensorflow as tf
from keras.models import load_model


In [ ]:
def probar_ModeloTransformers(val_glove, labels, tamano=0):
  
        

        
        max_timesteps = 9
        

        # Pad sequences to same length
        X_padded = pad_sequences(val_glove, maxlen=max_timesteps, dtype='float32', padding='post', truncating='post')

        # Create boolean mask: True where the sequence is non-zero (not padding)
        mask = (X_padded.sum(axis=-1) != 0)
    

        max_timesteps = 8  # Define the maximum length for padding

        
        # Cargar el modelo desde keras, como archivo local, agregando la función expand_mask como custom_objects
        def expand_mask(m):
            return tf.cast(tf.expand_dims(tf.expand_dims(m, 1), 1), tf.float32)
        model = load_model(f'creacionModelosBenchMark/modeloTransformers_{tamano}.keras',  custom_objects={'expand_mask': expand_mask})


        loss, acc = model.evaluate(
            {'input_embeddings': X_padded, 'input_mask': mask},
            labels,
            batch_size=16
        )
        print(f"Validation Accuracy: {acc:.4f}")
        return acc




In [130]:
probar_ModeloTransformers(embedding, data['ai'], tamano='chicos')
probar_ModeloTransformers(embedding, data['ai'], tamano='grandes')

9/9 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - accuracy: 0.6379 - loss: 0.5924
Validation Accuracy: 0.6232
9/9 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - accuracy: 0.9725 - loss: 0.0867
Validation Accuracy: 0.9565


0.95652174949646

# Guardar todos los resultados en un archivo .csv

In [131]:
# Obtener las predicciones de los modelos y guardar en un CSV
def obtener_predicciones(df, tamano=0):
    # Obtener las predicciones de los modelos y guardar en un CSV
    prediccionesLasso=probar_modeloLasso(imagenes, df['ai'], tamano=tamano)
    prediccionesLSTM=probar_modeloLSTM(embedding, df['ai'], tamano=tamano)
    prediccionesBERT=probar_modeloBert(df, tamano=tamano)
    prediccionesTransformers=probar_ModeloTransformers(embedding, df['ai'], tamano=tamano)
    prediccionesLassoLSTM=probar_modeloLasso_LSTM(imagenes, embedding, df['ai'], tamano=tamano)

    
  
    # Crear un dataframe para mostrar los resultados de accuracy con las columnas de los modelos
    df_resultados = pd.DataFrame({
        'Modelo': ['Lasso', 'LSTM', 'BERT', 'Transformers', 'LASSO+LSTM'],
        'Accuracy': [prediccionesLasso, prediccionesLSTM, prediccionesBERT, prediccionesTransformers, prediccionesLassoLSTM],
        'Tamano': [tamano]*5
    })

    
    # Si el archivo csv ya existe, se agrega al final
    try:
        df_existente = pd.read_csv('prediccionesBenchmark.csv')
        df_existente = pd.concat([df_existente, df_resultados], ignore_index=True)
        df_existente.to_csv('prediccionesBenchmark.csv', index=False)
    except FileNotFoundError:
        # Si el archivo no existe, se crea uno nuevo
        df_resultados.to_csv('prediccionesBenchmark.csv', index=False)
    

In [ ]:
obtener_predicciones(data, tamano='chicos')
obtener_predicciones(data, tamano='grandes')

Accuracy: 0.49


5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 103ms/step
Accuracy: 0.82
Validation Accuracy: 0.9275
9/9 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - accuracy: 0.6379 - loss: 0.5924
Validation Accuracy: 0.6232
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 110ms/step
Accuracy: 0.82
Accuracy: 0.78
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 153ms/step
Accuracy: 0.98
Validation Accuracy: 0.9783
9/9 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - accuracy: 0.9725 - loss: 0.0867
Validation Accuracy: 0.9565
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 103ms/step
Accuracy: 0.98


: 